### Setup & Imports

1. [Qwen2.5 Unsloth Fine-Tuning](https://colab.research.google.com/drive/1Kose-ucXO1IBaZq5BvbwWieuubP7hxvQ?usp=sharing#scrollTo=2eSvM9zX_2d3)
2. [LoRA Adabter](https://huggingface.co/A7med-Ame3/qwen2.5_lora_model)
3. [Base Model](https://huggingface.co/unsloth/Qwen2.5-7B-Instruct)
4. [Merged Model](https://huggingface.co/A7med-Ame3/Qwen2.5-7B-LiveKit-16bit)

In [2]:
# %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     !pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
#     !pip install --no-deps --upgrade "torchao>=0.16.0"
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

# !pip install --upgrade transformers

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret('HF_TOKEN')

!huggingface-cli login --token {HF_TOKEN}


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help



In [5]:
from unsloth import FastLanguageModel
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Load the Base Model

In [6]:
model_id = 'unsloth/Qwen2.5-7B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(model_id,
                                                     load_in_4bit=True,
                                                     max_seq_length = 2048,
                                                     dtype=None,
                                                     use_gradient_checkpointing='unsloth')

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.14.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [7]:
# model

### Load the Merged Model (LoRA + Base Model)

In [8]:
from unsloth import FastLanguageModel

merged_model, merged_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "A7med-Ame3/qwen2.5_lora_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.14.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.7.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
    (layers): ModuleList(
      (0-1): 2 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), 

### Model Evaluation & Benchmarking

#### Measure Some Metrics

1. `TTFT (Time-To-First-Token)`: The time taken from the start of request processing until the model outputs the first token. (A very important metric for real-time streaming).

2. `Total Inference Time`: The total time to generate the complete answer. 

3. `Generated Tokens Count`: The number of new tokens generated by the model.

4. `TPS (Tokens Per Second)`: Generation speed, measured as: $$\text{TPS} = \frac{\text{Generated Tokens Count}}{\text{Total Inference Time}}$$

In [9]:
from transformers import TextStreamer

class LatencyStreamer(TextStreamer):
    def __init__(self, tokenizer):
        super().__init__(tokenizer, skip_prompt=True)
        self.start_time = None
        self.first_token_time = None
        self.generated_tokens = 0

    def put(self, value):
        current_time = time.time()
        # Record First Time when token generated
        if self.first_token_time is None and value.numel() > 0:
            self.first_token_time = current_time
        
        # Number of Generated Tokens
        self.generated_tokens += value.numel()
        super().put(value)

streamer = LatencyStreamer(tokenizer)

#### Test Base Model

In [10]:
import time

FastLanguageModel.for_inference(model)
question = "ما الفرق بين النبي و الرسول ؟"

messages = [{
    'role': 'user',
    'content': question
}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must be added for Generation
    return_tensors='pt'
).to(model.device)

prompt_length = inputs.shape[1]

start_time = time.time()
streamer.start_time = start_time
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens=128,
    temperature=0.2,
    do_sample=True,
    streamer = TextStreamer(tokenizer, skip_prompt=True)
)
total_time = time.time() - start_time 

ttft = (streamer.first_token_time - start_time) if streamer.first_token_time else total_time
token_count = outputs[0].shape[0] - prompt_length
tps = token_count / total_time if total_time > 0 else 0

output = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


في 

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


الإسلام، يُستخدم المصطلحان "النبي" و"الرسول" بشكل متداخل، لكنهما لهما معانٍ متميزة:

1. النبي:
- هو الشخص الذي تحدث إليه الله مباشرة.
- أبلغ رسالته إلى قومه دون كتابة نصوص دينية.
- قد يكون من غير العرب.
- مثال: نوح، إبراهيم، موسى.

2. الرسول:
- هو شخص تكلم إليه الله وتلقى الوحي منه.
- أرسله الله لقومه برسالة محددة.



In [11]:
print("📊 INFERENCE BENCHMARK REPORT [Base Model]")
print("_"*40)
print(f"⏱️ Time-To-First-Token (TTFT) : {ttft:.4f} sec ({ttft*1000:.2f} ms)")
print(f"⏳ Total Inference Time        : {total_time:.4f} sec")
print(f"🔢 Total Output Tokens        : {token_count} tokens")
print(f"⚡ Tokens Per Second (TPS)     : {tps:.2f} tokens/sec")
print("_"*40)
print("\n📝 Output Answer:\n", output)

📊 INFERENCE BENCHMARK REPORT [Base Model]
________________________________________
⏱️ Time-To-First-Token (TTFT) : 12.0498 sec (12049.77 ms)
⏳ Total Inference Time        : 12.0498 sec
🔢 Total Output Tokens        : 128 tokens
⚡ Tokens Per Second (TPS)     : 10.62 tokens/sec
________________________________________

📝 Output Answer:
 في الإسلام، يُستخدم المصطلحان "النبي" و"الرسول" بشكل متداخل، لكنهما لهما معانٍ متميزة:

1. النبي:
- هو الشخص الذي تحدث إليه الله مباشرة.
- أبلغ رسالته إلى قومه دون كتابة نصوص دينية.
- قد يكون من غير العرب.
- مثال: نوح، إبراهيم، موسى.

2. الرسول:
- هو شخص تكلم إليه الله وتلقى الوحي منه.
- أرسله الله لقومه برسالة محددة.



#### Test the Merged Model

In [12]:
question = "ما الفرق بين النبي و الرسول ؟"

messages = [{
    'role': 'user',
    'content': question
}]

inputs = merged_tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must be added for Generation
    return_tensors='pt'
).to(merged_model.device)

prompt_length = inputs.shape[1]

start_time = time.time()
streamer.start_time = start_time
outputs = merged_model.generate(
    input_ids = inputs,
    max_new_tokens=128,
    temperature=0.2,
    do_sample=True,
    streamer = TextStreamer(merged_tokenizer, skip_prompt=True)
)
total_time = time.time() - start_time 

ttft = (streamer.first_token_time - start_time) if streamer.first_token_time else total_time
token_count = outputs[0].shape[0] - prompt_length
tps = token_count / total_time if total_time > 0 else 0

output = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


النبي 

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


هو اللي بيدعو للتوحيد وبيبلغ رسالته، لكن مش مكلف بإيصال كتاب موحى به. أما الرسول فده أعلى درجة، لأنه بيدعو لتوحيد الألوهية (عبادة ربنا لوحده) وتوحيد الربوبية (ربنا هو الخالق والرازق)، وبيكون مكلف بإيصال كتاب موحى به زي القرآن أو التوراة أو الإنجيل. يعني كل رسول نبي، بس مش كلنبي رسول.<|im_end|>


In [13]:
print("📊 INFERENCE BENCHMARK REPORT [Fine-Tuned Model]")
print("_"*40)
print(f"⏱️ Time-To-First-Token (TTFT) : {ttft:.4f} sec ({ttft*1000:.2f} ms)")
print(f"⏳ Total Inference Time        : {total_time:.4f} sec")
print(f"🔢 Total Output Tokens        : {token_count} tokens")
print(f"⚡ Tokens Per Second (TPS)     : {tps:.2f} tokens/sec")
print("_"*40)
print("\n📝 Output Answer:\n", output)

📊 INFERENCE BENCHMARK REPORT [Fine-Tuned Model]
________________________________________
⏱️ Time-To-First-Token (TTFT) : 8.0780 sec (8078.04 ms)
⏳ Total Inference Time        : 8.0780 sec
🔢 Total Output Tokens        : 121 tokens
⚡ Tokens Per Second (TPS)     : 14.98 tokens/sec
________________________________________

📝 Output Answer:
 النبي هو اللي بيدعو للتوحيد وبيبلغ رسالته، لكن مش مكلف بإيصال كتاب موحى به. أما الرسول فده أعلى درجة، لأنه بيدعو لتوحيد الألوهية (عبادة ربنا لوحده) وتوحيد الربوبية (ربنا هو الخالق والرازق)، وبيكون مكلف بإيصال كتاب موحى به زي القرآن أو التوراة أو الإنجيل. يعني كل رسول نبي، بس مش كلنبي رسول.


### Final Inference Function

In [14]:
from transformers import TextStreamer

def ask_sheikh(model, tokenizer, question, max_new_tokens=256, temperature=0.6, repetition_penalty=1.1, use_streamer=False):
    messages = [
        {
            'role': 'system',
            'content': (
                "أنت شيخ مصري حكيم وطيب، تتحدث بالعامية المصرية البسيطة والمحببة للقلب. "
                "ترد على الأسئلة الدينية بشكل مباشر وميسر ومبسط، وتستخدم عبارات طيبة ودعائية."
            )
        },
        {
            'role': 'user',
            'content': question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True
    ).to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True) if use_streamer else None

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,  
        repetition_penalty=repetition_penalty,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer
    )

    prompt_length = inputs['input_ids'].shape[1]
    output = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
    
    return output.strip()


In [15]:
### ---> Base Model
question = "كم عدد الصلوات فى اليوم الواحد ؟ و لمازا نصوم رمضان ؟"

output = ask_sheikh(model, tokenizer, question, use_streamer=False)

print("الرد النهائي:")
print("_"*20)
print(output)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


الرد النهائي:
____________________
الصلوات في اليوم الواحد هي خمس صلوات: الفجر، الظهر، العصر، المغرب، والعشاء.

و至于为什么我们斋月要封斋，简单来说是因为：
我们封斋是为了纪念先知穆罕默德在古莱什山洞中的启示之夜。另外，斋月也是为了培养我们的忍耐力、节制和同情心，因为通过自己挨饿受渴，我们可以体会到那些贫穷人生活的艰辛。所以，封斋不仅是宗教义务，也是一种美德的实践。
在伊斯兰教里，封斋还有助于人们更好地理解饥饿感，从而更加珍惜食物，并且激发帮助需要帮助的人的愿望。这样，斋月不仅仅是一个宗教仪式，它还是一种教育和社会行为的方式。

希望这个解释对你有帮助！愿你在这个神圣的月份中获得更多的恩典与祝福。阿门。


In [16]:
### ---> Merged Model
question = "كم عدد الصلوات فى اليوم الواحد ؟ و لمازا نصوم رمضان ؟"

output = ask_sheikh(merged_model, merged_tokenizer, question, use_streamer=False)

print("الرد النهائي:")
print("_"*20)
print(output)

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


الرد النهائي:
____________________
فيه خمس صلوات يومية: الفجر، والظهر، والعصر، والمغرب، والعشاء. أما الصيام فيرمضان فواجب على المسلم إنه ما ياكلش ولا يشربش من طلوع الفجر لحد غروب الشمس، ده عشان يتقرب لربنا ويستغفر ويحس بطعم الفقر اللي المسكين بياكله.


## Systematic Evaluation: Base vs Fine-Tuned

This section builds a **repeatable, quantitative** comparison between the base model (`unsloth/Qwen2.5-7B-Instruct` — swap in your actual base model id) and the fine-tuned model, covering:

- **Inference benchmarks**: TTFT, TPS, total time, peak VRAM — averaged over a held-out question set spanning all 4 intent categories.
- **Qualitative accuracy**: dialect consistency (Egyptian vs MSA marker heuristic) + a manual scoring template for slang understanding and intent-extraction accuracy.
- **Quantization loss**: semantic similarity between the full-precision merged model's answers and the GGUF `q4_k_m` quantized model's answers on the same prompts.

Run this after both models (base + fine-tuned) are loadable in this session, and after the GGUF file has been produced.

In [17]:
# 1) Held-out evaluation set — 16 questions across 4 intent categories
# Includes Egyptian slang / colloquial phrasing to stress-test dialect handling

eval_questions = [
    {"id": "aq1", "intent": "aqeedah", "question": "ما الفرق بين النبي و الرسول؟"},
    {"id": "aq2", "intent": "aqeedah", "question": "الايمان بالقضاء و القدر معناه ايه بالظبط؟"},
    {"id": "aq3", "intent": "aqeedah", "question": "ليه ربنا خلقنا مع انه عارف اننا هنعصي؟"},
    {"id": "aq4", "intent": "aqeedah", "question": "فيه ناس بتقول ان التوكل يعني متعملش حاجة، الكلام ده صح؟"},

    {"id": "fq1", "intent": "fiqh", "question": "لو نسيت اصلي ظهر و فاكر بعد العصر اعمل ايه؟"},
    {"id": "fq2", "intent": "fiqh", "question": "هل جايز اجمع صلاتين وانا مش مسافر بس تعبان؟"},
    {"id": "fq3", "intent": "fiqh", "question": "الزكاة بتتحسب ازاي على الفلوس اللي في البنك؟"},
    {"id": "fq4", "intent": "fiqh", "question": "بصراحه مش فاهم الفرق بين السنة المؤكدة والسنة الغير مؤكدة"},

    {"id": "hs1", "intent": "history", "question": "احكيلي باختصار قصة هجرة الرسول من مكة للمدينة"},
    {"id": "hs2", "intent": "history", "question": "مين هما الخلفاء الراشدين بالترتيب؟"},
    {"id": "hs3", "intent": "history", "question": "غزوة بدر كانت في اي سنة وليه كانت مهمة كده؟"},
    {"id": "hs4", "intent": "history", "question": "ايه قصة اصحاب الكهف باختصار؟"},

    # general / conversational (heavy slang, to stress-test dialect + slang understanding)
    {"id": "gn1", "intent": "general", "question": "يا شيخ انا حاسس اني تعبان نفسيا اوي، ايه رأي الدين في الموضوع ده؟"},
    {"id": "gn2", "intent": "general", "question": "صحابي بيقولي كل حاجة مكتوبة يبقى مفيش داعي اجتهد، رد عليه ازاي؟"},
    {"id": "gn3", "intent": "general", "question": "امتى بقى العلم واجب على كل مسلم؟"},
    {"id": "gn4", "intent": "general", "question": "ايه حكم اني اسيب صلاة الجمعة عشان شغل مستعجل؟"},
]

print(f"Loaded {len(eval_questions)} evaluation questions across "
      f"{len(set(q['intent'] for q in eval_questions))} intent categories.")

Loaded 16 evaluation questions across 4 intent categories.


In [18]:
# 2) Benchmark harness: TTFT, TPS, total time, peak VRAM
# Works for any (model, tokenizer) pair loaded via transformers/Unsloth

import time
import torch
import pandas as pd
from threading import Thread
from transformers import TextIteratorStreamer


def benchmark_generate(model, tokenizer, question, max_new_tokens=200, temperature=0.3):
    """Runs one generation and returns TTFT, total_time, TPS, output_tokens, peak VRAM (MB), and the text."""

    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    gen_kwargs = dict(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        streamer=streamer,
    )

    start_time = time.time()
    first_token_time = None
    output_text = ""
    token_count = 0

    thread = Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()

    for chunk in streamer:
        if first_token_time is None and chunk.strip() != "":
            first_token_time = time.time()
        output_text += chunk
        token_count += 1

    thread.join()
    end_time = time.time()

    if torch.cuda.is_available():
        torch.cuda.synchronize()
        peak_vram_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        peak_vram_mb = None

    # Recompute exact output token count via tokenizer (streamer yields text chunks, not 1:1 with tokens)
    real_token_count = len(tokenizer(output_text, add_special_tokens=False)["input_ids"])

    total_time = end_time - start_time
    ttft = (first_token_time - start_time) if first_token_time else total_time
    tps = real_token_count / total_time if total_time > 0 else 0.0

    return {
        "ttft_sec": round(ttft, 3),
        "total_time_sec": round(total_time, 3),
        "output_tokens": real_token_count,
        "tps": round(tps, 2),
        "peak_vram_mb": round(peak_vram_mb, 1) if peak_vram_mb else None,
        "output_text": output_text.strip(),
    }


def run_benchmark(model, tokenizer, questions, model_label, max_new_tokens=200):
    rows = []
    for q in questions:
        result = benchmark_generate(model, tokenizer, q["question"], max_new_tokens=max_new_tokens)
        rows.append({
            "model": model_label,
            "id": q["id"],
            "intent": q["intent"],
            "question": q["question"],
            **result,
        })
        print(f"[{model_label}] {q['id']} ({q['intent']}) — TTFT {result['ttft_sec']}s, "
              f"TPS {result['tps']}, tokens {result['output_tokens']}, "
              f"VRAM {result['peak_vram_mb']}MB")
    return pd.DataFrame(rows)

### Run the benchmark on the base model

In [19]:
base_results_df = run_benchmark(model, tokenizer, eval_questions, model_label="base")

base_results_df

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[base] aq1 (aqeedah) — TTFT 0.273s, TPS 13.57, tokens 200, VRAM 14155.9MB


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[base] aq2 (aqeedah) — TTFT 0.293s, TPS 13.23, tokens 170, VRAM 14156.8MB


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[base] gn2 (general) — TTFT 0.275s, TPS 15.16, tokens 200, VRAM 14158.0MB


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[base] gn3 (general) — TTFT 0.276s, TPS 14.81, tokens 177, VRAM 14156.4MB
[base] gn4 (general) — TTFT 0.21s, TPS 14.96, tokens 200, VRAM 14157.3MB


,model,id,intent,question,ttft_sec,total_time_sec,output_tokens,tps,peak_vram_mb,output_text
0,base,aq1,aqeedah,ما الفرق بين النبي و الرسول؟,0.273,14.742,200,13.57,14155.9,"في الإسلام، يُستخدم المصطلحان ""النبي"" و""الرسول..."
1,base,aq2,aqeedah,الايمان بالقضاء و القدر معناه ايه بالظبط؟,0.293,12.854,170,13.23,14156.8,الإيمان بالقضاء والقدر هو أحد أهم العقائد الإس...
2,base,aq3,aqeedah,ليه ربنا خلقنا مع انه عارف اننا هنعصي؟,0.355,13.606,199,14.63,14156.9,هذا سؤال عقلي عميق ومتعدد الآراء. هناك العديد ...
3,base,aq4,aqeedah,فيه ناس بتقول ان التوكل يعني متعملش حاجة، الكل...,0.267,10.962,169,15.42,14157.3,لا، هذا القول غير صحيح. التوكل في الإسلام ليس ...
4,base,fq1,fiqh,لو نسيت اصلي ظهر و فاكر بعد العصر اعمل ايه؟,0.265,12.596,199,15.80,14157.3,إذا نسيت صلاتك الظهر وانتقلت إلى صلاة العصر، ي...
5,base,fq2,fiqh,هل جايز اجمع صلاتين وانا مش مسافر بس تعبان؟,0.199,12.669,200,15.79,14157.3,في الإسلام، إذا كان الشخص متعبًا أو مريضًا ولي...
6,base,fq3,fiqh,الزكاة بتتحسب ازاي على الفلوس اللي في البنك؟,0.269,13.137,200,15.22,14156.8,لحساب الزكاة على الأموال المودعة في البنك، يجب...
7,base,fq4,fiqh,بصراحه مش فاهم الفرق بين السنة المؤكدة والسنة ...,0.341,13.007,190,14.61,14157.3,بالطبع، سأحاول توضيح الفرق بين السنة المؤكدة و...
8,base,hs1,history,احكيلي باختصار قصة هجرة الرسول من مكة للمدينة,0.344,13.661,200,14.64,14156.8,هجرة الرسول محمد صلى الله عليه وسلم من مكة إلى...
9,base,hs2,history,مين هما الخلفاء الراشدين بالترتيب؟,0.339,8.876,130,14.65,14156.3,الخلفاء الراشدون هم أول خلفاء للمسلمين بعد وفا...


### Run the benchmark on the fine-tuned model

In [20]:
finetuned_results_df = run_benchmark(merged_model, merged_tokenizer, eval_questions, model_label="finetuned")

finetuned_results_df

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=2

[finetuned] aq1 (aqeedah) — TTFT 0.373s, TPS 14.46, tokens 114, VRAM 14167.0MB


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[finetuned] aq2 (aqeedah) — TTFT 0.364s, TPS 14.43, tokens 130, VRAM 14167.9MB


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[finetuned] gn2 (general) — TTFT 0.363s, TPS 14.6, tokens 90, VRAM 14169.1MB


Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[finetuned] gn3 (general) — TTFT 0.362s, TPS 14.61, tokens 86, VRAM 14167.5MB
[finetuned] gn4 (general) — TTFT 0.363s, TPS 14.63, tokens 97, VRAM 14168.4MB


,model,id,intent,question,ttft_sec,total_time_sec,output_tokens,tps,peak_vram_mb,output_text
0,finetuned,aq1,aqeedah,ما الفرق بين النبي و الرسول؟,0.373,7.884,114,14.46,14167.0,النبي هو اللي بيجي برسالة من ربنا، لكن مش مكلف...
1,finetuned,aq2,aqeedah,الايمان بالقضاء و القدر معناه ايه بالظبط؟,0.364,9.010,130,14.43,14167.9,الإيمان بالقضاء والقدر معناه إن المسلم يصدق إن...
2,finetuned,aq3,aqeedah,ليه ربنا خلقنا مع انه عارف اننا هنعصي؟,0.372,6.215,89,14.32,14168.0,ربنا سبحانه وتعالى عارف كل حاجة هتحصل قبل ما ت...
3,finetuned,aq4,aqeedah,فيه ناس بتقول ان التوكل يعني متعملش حاجة، الكل...,0.442,10.010,146,14.58,14168.4,لأ، الكلام ده غلط تمامًا ومش فاهم للتوكل الصح....
4,finetuned,fq1,fiqh,لو نسيت اصلي ظهر و فاكر بعد العصر اعمل ايه؟,0.305,3.951,56,14.18,14168.4,لو نسيت اصلي ظهر وفاكر بعد ما صليت العصر، لازم...
5,finetuned,fq2,fiqh,هل جايز اجمع صلاتين وانا مش مسافر بس تعبان؟,0.426,9.067,133,14.67,14168.4,لأ، مينفعش تجمع بين صلاة فرض وصلاة فرض، زي ما ...
6,finetuned,fq3,fiqh,الزكاة بتتحسب ازاي على الفلوس اللي في البنك؟,0.499,8.469,124,14.64,14167.9,الزكاة بتكون على الفلوس اللي عندك لو بلغت النص...
7,finetuned,fq4,fiqh,بصراحه مش فاهم الفرق بين السنة المؤكدة والسنة ...,0.368,7.163,105,14.66,14168.4,الفرق الأساسي إن السنة المؤكدة ليها دليل قطعي ...
8,finetuned,hs1,history,احكيلي باختصار قصة هجرة الرسول من مكة للمدينة,0.427,8.758,129,14.73,14167.9,هجرة الرسول صلى الله عليه وسلم كانت في السنة ا...
9,finetuned,hs2,history,مين هما الخلفاء الراشدين بالترتيب؟,0.424,4.979,72,14.46,14167.4,الخلفاء الراشدين دول أربعة: أبو بكر الصديق، ثم...


### Aggregate benchmark results

In [21]:
all_results_df = pd.concat([base_results_df, finetuned_results_df], ignore_index=True)

overall_summary = all_results_df.groupby("model")[["ttft_sec", "total_time_sec", "tps", "output_tokens", "peak_vram_mb"]].mean().round(2)
display(overall_summary)

,ttft_sec,total_time_sec,tps,output_tokens,peak_vram_mb
model,,,,,
base,0.29,12.75,14.89,189.62,14156.96
finetuned,0.39,7.68,14.58,112.25,14168.06


In [22]:
per_intent_summary = all_results_df.groupby(["model", "intent"])[["ttft_sec", "tps", "output_tokens"]].mean().round(2)
display(per_intent_summary)

ttft_sec    tps  output_tokens
model     intent                                 
base      aqeedah      0.30  14.21         184.50
          fiqh         0.27  15.36         197.25
          general      0.26  15.06         194.25
          history      0.32  14.92         182.50
finetuned aqeedah      0.39  14.45         119.75
          fiqh         0.40  14.54         104.50
          general      0.36  14.67         110.75
          history      0.41  14.66         114.00